Aprendizado de variedades (Manifold Learning) com fluxo de clusterização distribuída por tipo de característica é mais informativo do que o UMAP para conjuntos de dados clínicos tabulares

Importando as Libraries

In [ ]:
import numpy as np                                #para operações matemáticas e processamento de matrizes(Euclidiana, Hamming e a Canberra Modificada)
import pandas as pd                               #manipulação de dados tabulares
from sklearn.preprocessing import StandardScaler  #escalonamento de dados para garantir que todas as variáveis contribuam igualmente
import matplotlib.pyplot as plt                   #para gerar todos os gráficos e mapas de calor
import seaborn as sns                             
%matplotlib inline                                #para exibir os gráficos diretamente abaixo da celula
import warnings                                   #suprimir avisos de sistema durante a execução
warnings.filterwarnings('ignore')                 
import tensorflow as tf                           #bibliotecas de Deep Learning
from tensorflow import keras                      #Facilita criar redes neurais
import umap.umap_ as umap                         #importa o umap usado
%config InlineBackend.figure_format = 'svg'

In [ ]:
#Importa os algorimos e medidas usadas no processo
from fdc.fdc import feature_clustering
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified
from fdc.clustering import *

from sklearn import metrics
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score
from cluster_val import *

Importando os dados pré-processados

In [ ]:
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
data=pd.read_csv('heart_failure_clinical_records_dataset.csv')

In [ ]:
np.random.seed(42)
data=data.sample(frac=1) #Embaralha o dataset
np.random.seed(42)
i=[x for x in range(299)]

data.set_index(pd.Series(i), inplace=True)

In [ ]:
data.drop('DEATH_EVENT',axis=1,inplace=True)

UMAP nos dados originais

In [ ]:
#Importa funções e classes do FDC
from fdc.fdc import feature_clustering
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified
from fdc.clustering import *

In [ ]:
umap_emb=feature_clustering(15,0.1,'euclidean',data,True)

ANOVA test, Silhouette_score e Dunn index para os clusters do UMAP extraidos usando clustering K-means

In [ ]:
umap_clustering=Clustering(umap_emb,umap_emb,True)
umap_cluster_list,umap_cluster_counts=umap_clustering.K_means(3)

In [ ]:
cont_list= ['age','creatinine_phosphokinase','ejection_fraction','platelets','serum_creatinine','serum_sodium','time']

ord_list= ['anaemia','diabetes','high_blood_pressure','sex','smoking']

nom_list= []

In [ ]:
### ANOVA TEST 

data['Cluster']=umap_cluster_list
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))):#add -1 if there is noise otherwise not necessary------applicable only for DBSCAN
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
silhouette_score(umap_emb, umap_cluster_list, metric='euclidean')

Visualizar Silhouette score (Pode-se escolher o números de clusters baseados no resultado)

In [ ]:
Silhouette_visual(umap_emb)

In [ ]:
elbow_plot(umap_emb)

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list))

ANOVA test, Silhouette_score e Dunn index para os clusters do UMAP extraidos usando clustering Aglomerativo

In [ ]:
umap_cluster_list_agglo,umap_cluster_counts_agglo=umap_clustering.Agglomerative(3,'euclidean','ward')

In [ ]:
### ANOVA TEST

data['Cluster']=umap_cluster_list_agglo
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))):
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_agglo))

ANOVA test, Silhouette_score e Dunn index para os clusters do UMAP extraidos usando clustering DBSCAN 

In [ ]:
umap_cluster_list_dbscan,umap_cluster_counts_dbscan=umap_clustering.DBSCAN(0.9,20)

In [ ]:
### ANOVA TEST

data['Cluster']=umap_cluster_list_dbscan
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))):#add -1 if there is noise otherwise not necessary------applicable only for DBSCAN
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2,c_3)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
#Remove os indices considerados ruídos pelo algoritmo
non_noise_indices= np.where(np.array(umap_cluster_list_dbscan)!=-1)
umap_emb= umap_emb.iloc[non_noise_indices]
#FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
umap_cluster_list_dbscan= np.array(umap_cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_dbscan))

Dividindo as variaveis
- cont_list = contínuas
- ord_list = ordinais

In [ ]:
cont_list= ['age','creatinine_phosphokinase','ejection_fraction','platelets','serum_creatinine','serum_sodium','time']

ord_list= ['anaemia','diabetes','high_blood_pressure','sex','smoking']

In [ ]:
len(ord_list)

In [ ]:
len(cont_list)

Aplicando FDC nos dados originais

In [ ]:
from fdc.fdc import feature_clustering
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified

In [ ]:
fdc = FDC(clustering_cont=Clustering('euclidean',15,0.1) #número de vizinhos e parâmetro de densidade
          , clustering_ord=Clustering(modified_can,15,0.1,max_components=1) #usa Canberra modificada que é mais sensível a valores pequenos
          , visual=True
          , use_pandas_output=True
          , with_2d_embedding=True #projeção 2D
          )

fdc.selectFeatures(continueous=cont_list, ordinal=ord_list) #diz ao modelo quais variáveis são contínuas e quais são ordinais

FDC_emb_high,FDC_emb_low = fdc.normalize(data,
                                        n_neighbors=15,         #numero de vizinhos
                                        min_dist=0.1,           #o quão proximo os clusters vão ficar
                                        cont_list=cont_list,
                                        ord_list=ord_list,
                                        with_2d_embedding=True, #Gera duas representações, uma de alta e outra de baixa dimensão
                                        visual=True)

ANOVA test, Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando k-means

In [ ]:
from fdc.clustering import Clustering

In [ ]:
FDC_emb_low.rename(columns={"UMAP_0": "FDC_0", "UMAP_1": "FDC_1"},inplace=True)

In [ ]:
clustering=Clustering(FDC_emb_low,FDC_emb_low,True)
cluster_list,cluster_counts=clustering.K_means(6)

In [ ]:
### ANOVA TEST

data['Cluster']=cluster_list
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))):#add -1 if there is noise otherwise not necessary------applicable only for DBSCAN
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2,c_3,c_4,c_5)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
silhouette_score(FDC_emb_low, cluster_list, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_low,cluster_list))

Visualizando Shilhouette score para FDC embedding de baixa dimensão

In [ ]:
Silhouette_visual(FDC_emb_low)

Elbow plot para FDC embedding de baixa dimensão

In [ ]:
elbow_plot(FDC_emb_low)

ANOVA test, Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering aglomerativo

In [ ]:
cluster_list_agglo,cluster_counts_agglo=clustering.Agglomerative(6,'euclidean','ward')

In [ ]:
### ANOVA TEST

data['Cluster']=cluster_list_agglo
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))):#add -1 if there is noise otherwise not necessary------applicable only for DBSCAN
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2,c_3,c_4,c_5)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
silhouette_score(FDC_emb_low, cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_low,cluster_list_agglo))

ANOVA test, Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering do DBSCAN

In [ ]:
cluster_list_dbscan,cluster_counts_dbscan=clustering.DBSCAN(0.9,20)

In [ ]:
### ANOVA TEST


data['Cluster']=cluster_list_dbscan
test_results={}
c_names=data.columns
from scipy.stats import f_oneway
for i in c_names[:-1]:
    for j in range(len(np.unique(data.Cluster))-1):#add -1 if there is noise otherwise not necessary------applicable only for DBSCAN
        var_name="c_"+str(j)
        myVars = vars()
        myVars[var_name]=list(data[i][data["Cluster"]==j])
    
    stats,p_val=f_oneway(c_0,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8)
    test_results[i]=stats,float("{:.4f}".format(p_val))

test_results=pd.DataFrame.from_dict(test_results, orient='index', columns=["stats","p_value"])
test_column_list=test_results.index[test_results["p_value"]<0.05].to_list()
ord_count=nom_count=cont_count=0
for i in test_column_list:
    if i in ord_list:
        ord_count+=1
    elif i in nom_list:
        nom_count+=1
    elif i in cont_list:
        cont_count+=1

In [ ]:
cluster_counts_dbscan

In [ ]:
#removing noise indices from the embeddings
non_noise_indices= np.where(np.array(cluster_list_dbscan)!=-1)
FDC_emb_high= FDC_emb_high.iloc[non_noise_indices]
FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
cluster_list_dbscan= np.array(cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(FDC_emb_low, cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_low,cluster_list_dbscan))